# Experiments Notebook

This notebook is used for running experiments and testing different configurations of the neural network.

In [1]:
import sys
import torch
from pathlib import Path

# Get the current working directory and check where we are
print(f"Current working directory: {Path.cwd()}")

# If you're in the notebooks directory, go up one level
if Path.cwd().name == 'notebooks':
    sys.path.append(str(Path.cwd().parent))
else:
    # If you're in the project root, add current directory
    sys.path.append(str(Path.cwd()))

# Import necessary libraries

from src.layers.input import InputLayer;
from src.layers.dense import DenseLayer;

from src.losses.cross_entropy import Loss;
from src.optimizers.gradient_descent import GD;
from src.utils.data_loader import load_data;
from src.trainers.trainer import Trainer



Current working directory: /home/protim/Documents/basis/notebooks


In [2]:
# Define hyperparameters
hyperparams = {
    'input_units': 2,
    'hidden_units': 2,
    'output_units': 1

}

In [3]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_layer = InputLayer( hyperparams, name='input', device=device );
hidden_layer = DenseLayer( hyperparams, name='hidden', transfer='sigmoid', device=device );
output_layer = DenseLayer( hyperparams, name='output', transfer='sigmoid', device=device );

network = input_layer >> hidden_layer >> output_layer;

In [4]:
# Load XOR data
( train_data, train_labels ), ( test_data, test_labels ) = load_data( "xor", device=device );
print( f"Train data shape: {train_data.shape}, Train labels shape: {train_labels.shape}" );

Train data shape: torch.Size([4, 2]), Train labels shape: torch.Size([4, 1])


In [5]:
print( f"\nNumber of trainable parameters: {sum( p.numel() for p in network.trainable_parameters.values() )}" );
print( f"\nTrainable Parameters:" );
for name, param in network.trainable_parameters.items():
    print( f"{name}: {param.shape}" );

# print param values
print("\nInitial parameter values:");
for name, param in network.trainable_parameters.items():
    print(f"{name}: {param.data}")



Number of trainable parameters: 9

Trainable Parameters:
hidden: torch.Size([2, 1])
W_input_hidden_layer: torch.Size([2, 2])
W_hidden_output_layer: torch.Size([1, 2])
output: torch.Size([1, 1])

Initial parameter values:
hidden: tensor([[0.],
        [0.]], device='cuda:0')
W_input_hidden_layer: tensor([[ 0.1372,  1.5283],
        [-0.1217,  0.6004]], device='cuda:0')
W_hidden_output_layer: tensor([[ 0.1136, -0.0884]], device='cuda:0')
output: tensor([[0.]], device='cuda:0')


In [6]:
print("XOR Network Forward Pass:")
for i in range(train_data.shape[0]):
    input_tensor = train_data[i].unsqueeze(1) # Shape (2, 1)
    output = network.forward(input_tensor)
    print(f"Input: {train_data[i].tolist()} -> Output: {output['output'].squeeze().item():.4f}, Expected: {train_labels[i].item():.1f}")

XOR Network Forward Pass:
Input: [0.0, 0.0] -> Output: 0.5032, Expected: 0.0
Input: [0.0, 1.0] -> Output: 0.5091, Expected: 1.0
Input: [1.0, 0.0] -> Output: 0.5048, Expected: 1.0
Input: [1.0, 1.0] -> Output: 0.5102, Expected: 0.0


In [7]:
# 1. Instantiate Loss Function
bce_loss = Loss(name="binary_cross_entropy")

In [8]:
# 2. Compile the network (Optimizer and Learning Rate)
learning_rate = 0.5 # You might need to tune this
network.compile(loss=bce_loss, optimizer_class=GD, learning_rate=learning_rate)

In [9]:
# 3. Train the Network
epochs = 5000 # More epochs for XOR
import threading
# Start GPU monitoring in separate thread
print("\nStarting Training...")
network.train(train_data, train_labels, epochs=epochs, save_param_history=True)
print("Training Finished.")


Starting Training...
GPU Available: NVIDIA GeForce RTX 3080
Number of GPUs: 1
TensorBoard log directory created at: runs/nn_training_gpu_cuda/run_20250615_151010_fa6643dc


/home/protim/Documents/basis/src/models/neural_network.py:91: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  if param_tensor.grad is not None:
/home/protim/Documents/basis/src/layers/weight.py:101: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you acce

Epoch 1/5000, Loss: 0.693021, Time: 126.9ms
Epoch 100/5000, Loss: 0.664123, Time: 25.2ms
Epoch 200/5000, Loss: 0.446230, Time: 21.3ms
Epoch 300/5000, Loss: 0.080884, Time: 9.6ms
Epoch 400/5000, Loss: 0.032136, Time: 9.4ms
Epoch 500/5000, Loss: 0.019613, Time: 11.3ms
Epoch 600/5000, Loss: 0.014024, Time: 9.5ms
Epoch 700/5000, Loss: 0.010883, Time: 30.4ms
Epoch 800/5000, Loss: 0.008878, Time: 11.9ms
Epoch 900/5000, Loss: 0.007489, Time: 9.0ms
Epoch 1000/5000, Loss: 0.006472, Time: 11.5ms
Epoch 1100/5000, Loss: 0.005696, Time: 23.5ms
Epoch 1200/5000, Loss: 0.005084, Time: 8.0ms
Epoch 1300/5000, Loss: 0.004590, Time: 9.0ms
Epoch 1400/5000, Loss: 0.004182, Time: 8.0ms
Epoch 1500/5000, Loss: 0.003841, Time: 18.3ms
Epoch 1600/5000, Loss: 0.003550, Time: 12.9ms
Epoch 1700/5000, Loss: 0.003300, Time: 11.4ms
Epoch 1800/5000, Loss: 0.003083, Time: 9.1ms
Epoch 1900/5000, Loss: 0.002892, Time: 15.6ms
Epoch 2000/5000, Loss: 0.002724, Time: 11.5ms
Epoch 2100/5000, Loss: 0.002574, Time: 8.2ms
Epoch 22

In [20]:
# Load TensorBoard extension
%load_ext tensorboard

# Display TensorBoard in the notebook
%tensorboard --logdir ../notebooks/runs/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6008 (pid 849050), started 0:02:13 ago. (Use '!kill 849050' to kill it.)